In [ ]:
import os
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()

# Connection details
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
AUTH = (os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "password"))
DB = os.getenv("NEO4J_DB", "nvd")

def create_lateral_movement_data():
    query = """
    // Create a "Crown Jewel" Internal Database
    MERGE (db:Application {name: 'CoreBankingDB', tier: 'P0'})
    MERGE (db_ins:ComputeInstance {id: 'i-9999', name: 'prod-db-internal', private_ip: '10.0.5.1'})
    MERGE (db)-[:HOSTED_ON]->(db_ins)
    
    // Transition from MERGE to MATCH requires WITH
    WITH db_ins
    
    // Create an Internal Network Path
    // Scenario A server (gateway) can reach the Scenario B server (worker)
    MATCH (insA:ComputeInstance {id: 'i-0001'})
    MATCH (insB:ComputeInstance {id: 'i-0002'})
    MERGE (insA)-[:CAN_REACH {port: 8080}]->(insB)
    
    WITH insB, db_ins
    
    // Scenario B server (worker) can reach the Crown Jewel DB
    MERGE (insB)-[:CAN_REACH {port: 5432}]->(db_ins)
    """
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session(database=DB) as session:
            session.run(query)
    print("Lateral movement paths successfully added to the VPEM graph.")

In [2]:
create_lateral_movement_data()

CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: WITH is required between MERGE and MATCH (line 9, column 5 (offset: 370))
"    MATCH (insA:ComputeInstance {name: 'api-gateway-01'})"
     ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}